# MCS test — gold vs cross-asset implied vol, δ=0.999, 6-year window

Is **cross-asset** implied vol (equity VIX, crude OVX) good enough on its own to
forecast gold realized vol, or is **gold-specific** implied vol (GVZ) genuinely better?
And do any of these beat a plain macro-release baseline?

6-model Model Confidence Set (Hansen, Lunde & Nason 2011), all fixed at recency weight
δ=0.999 and a 6-year rolling window. Helper code is copied verbatim from
`HAR_simpleOLS_3d_with_macro.ipynb` (Cells 1–3, 8). Results saved to `iv_comparison.csv`.

| Model | Extra features | Implied vol? | Macro? |
|-------|----------------|--------------|--------|
| log HAR + GVZ              | `log_GVZ`               | gold   | no  |
| log HAR + SPX IV (VIX)     | `log_VIX`               | equity | no  |
| log HAR + Crude IV (OVX)   | `log_OVX`               | crude  | no  |
| log HAR + Macro            | `macro`                 | no     | yes |
| log HAR + Macro + SPX RV   | `macro, log_RV_ES`      | no     | yes |
| log HAR + Macro + Crude RV | `macro, log_RV_crude`   | no     | yes |

In [1]:
# ===========================================================================
# Cell 1 — Imports & data (+ fixed test parameters)
# ===========================================================================
import numpy as np
import pandas as pd

# RV/GVZ panel merged with the scheduled-macro-release indicator (macro_event)
data = pd.read_parquet("merged_RV_GVZ_with_macro_event.parquet")
rv = data["RV_gold"].astype(float)

TRADING_DAYS = 252
EPS = 1e-6                                  # QLIKE forecast floor

# Two fixed parameters for this head-to-head test
WINDOW = int(round(6.0 * TRADING_DAYS))     # 1512 trading days (6 years)
DELTA  = 0.999                              # geometric recency-decay factor

print(f"RV_gold: {len(rv)} obs, {rv.index.min().date()} .. {rv.index.max().date()}")
print(f"Columns available: {list(data.columns)}")
print(f"macro_event release days in sample: {int(data['macro_event'].sum())} of {len(data)}")
print(f"WINDOW = {WINDOW} days (~6y),  DELTA = {DELTA}")

RV_gold: 4015 obs, 2010-06-08 .. 2026-05-29
Columns available: ['RV_gold', 'RV_crude', 'RV_ES', 'GVZ_close', 'macro_event']
macro_event release days in sample: 649 of 4015
WINDOW = 1512 days (~6y),  DELTA = 0.999


In [2]:
# ===========================================================================
# Cell 2 — Build the six design tables
# ===========================================================================
# Log-HAR components on x = log(RV_gold):
#   x_d[t]=x_t, x_w[t]=mean(x_{t-4..t}), x_m[t]=mean(x_{t-21..t})
#   y_log[t]=x_{t+1}=log(RV_{t+1}); y_level[t]=RV_{t+1} (kept for QLIKE in levels).
# Exogenous day-t terms are logged and known at the close (no look-ahead). The macro
# indicator is a RAW 0/1 dummy (never logged), shifted to day t+1: macro_event is a
# scheduled release flag known in advance, so the t+1 value is known at day t.

# Strict positivity required for every logged input (macro_event is a 0/1 dummy, excluded)
for col in ["RV_gold", "GVZ_close", "RV_ES", "RV_crude"]:
    assert (data[col] > 0).all(), f"{col} has non-positive values; log undefined"

x = np.log(rv)

def build_log_design(extra_cols):
    """Base log-HAR design + optional exogenous day-t columns (dict name->series)."""
    df = pd.DataFrame(index=rv.index)
    df["x_d"] = x
    df["x_w"] = x.rolling(5).mean()
    df["x_m"] = x.rolling(22).mean()
    for name, series in extra_cols.items():
        df[name] = series.reindex(rv.index)
    df["y_log"]   = x.shift(-1)        # log(RV_{t+1})
    df["y_level"] = rv.shift(-1)       # RV_{t+1} in levels, for QLIKE
    return df.dropna()

log_gvz   = np.log(data["GVZ_close"])
log_spx   = np.log(data["RV_ES"])
log_crude = np.log(data["RV_crude"])
macro     = data["macro_event"].shift(-1).astype(float)   # day-(t+1) scheduled-release dummy

# Cross-asset implied vol (VIX = equity IV, OVX = crude IV), aligned to the RV index.
# build_log_design reindexes each series onto rv.index; cross_asset_iv covers every RV
# date, so no extra rows are dropped and these designs share the others' index.
iv = pd.read_parquet("cross_asset_iv.parquet")
for col in ["VIX_close", "OVX_close"]:
    assert (iv[col] > 0).all(), f"{col} has non-positive values; log undefined"
log_vix = np.log(iv["VIX_close"])
log_ovx = np.log(iv["OVX_close"])

# Six designs: three IV-only specs (gold / equity / crude) vs three macro specs.
d_gvz         = build_log_design({"log_GVZ": log_gvz})                       # gold  IV only, NO macro
d_vix         = build_log_design({"log_VIX": log_vix})                       # equity IV only, NO macro/GVZ
d_ovx         = build_log_design({"log_OVX": log_ovx})                       # crude  IV only, NO macro/GVZ
d_macro       = build_log_design({"macro": macro})                          # macro only, NO IV
d_macro_spx   = build_log_design({"macro": macro, "log_RV_ES": log_spx})
d_macro_crude = build_log_design({"macro": macro, "log_RV_crude": log_crude})

for name, df in [("d_gvz", d_gvz), ("d_vix", d_vix), ("d_ovx", d_ovx), ("d_macro", d_macro),
                 ("d_macro_spx", d_macro_spx), ("d_macro_crude", d_macro_crude)]:
    assert df.notna().all().all(), f"unexpected NaNs in {name}"
    if "macro" in df.columns:
        assert set(np.unique(df["macro"].to_numpy())) <= {0.0, 1.0}, f"{name}: macro not 0/1"

# All six designs share the same dropna pattern, so their indices must be identical
for name, df in [("d_vix", d_vix), ("d_ovx", d_ovx), ("d_macro", d_macro),
                 ("d_macro_spx", d_macro_spx), ("d_macro_crude", d_macro_crude)]:
    assert d_gvz.index.equals(df.index), f"design index of {name} misaligned with d_gvz"

print("log HAR + GVZ              cols:", list(d_gvz.columns),         f"({len(d_gvz)} rows)")
print("log HAR + SPX IV (VIX)     cols:", list(d_vix.columns),         f"({len(d_vix)} rows)")
print("log HAR + Crude IV (OVX)   cols:", list(d_ovx.columns),         f"({len(d_ovx)} rows)")
print("log HAR + Macro            cols:", list(d_macro.columns),       f"({len(d_macro)} rows)")
print("log HAR + Macro + SPX RV   cols:", list(d_macro_spx.columns),   f"({len(d_macro_spx)} rows)")
print("log HAR + Macro + Crude RV cols:", list(d_macro_crude.columns), f"({len(d_macro_crude)} rows)")
d_gvz.head()

log HAR + GVZ              cols: ['x_d', 'x_w', 'x_m', 'log_GVZ', 'y_log', 'y_level'] (3993 rows)
log HAR + SPX IV (VIX)     cols: ['x_d', 'x_w', 'x_m', 'log_VIX', 'y_log', 'y_level'] (3993 rows)
log HAR + Crude IV (OVX)   cols: ['x_d', 'x_w', 'x_m', 'log_OVX', 'y_log', 'y_level'] (3993 rows)
log HAR + Macro            cols: ['x_d', 'x_w', 'x_m', 'macro', 'y_log', 'y_level'] (3993 rows)
log HAR + Macro + SPX RV   cols: ['x_d', 'x_w', 'x_m', 'macro', 'log_RV_ES', 'y_log', 'y_level'] (3993 rows)
log HAR + Macro + Crude RV cols: ['x_d', 'x_w', 'x_m', 'macro', 'log_RV_crude', 'y_log', 'y_level'] (3993 rows)


,x_d,x_w,x_m,log_GVZ,y_log,y_level
Date,,,,,,
2010-07-08,2.702348,2.887899,2.773932,3.008648,2.723668,15.236102
2010-07-09,2.723668,2.800490,2.762134,2.979603,2.691347,14.751526
2010-07-12,2.691347,2.766723,2.755241,2.918311,2.593328,13.374209
2010-07-13,2.593328,2.694457,2.744754,2.943913,2.754495,15.713097
2010-07-14,2.754495,2.693037,2.755626,2.903617,2.722249,15.214503


In [3]:
# ===========================================================================
# Cell 3 — Helpers: QLIKE, recency weights, BATCHED WLS per-day loss series
# ===========================================================================
# Common out-of-sample period from the 6-year warm-up. All four designs share one
# index, so a single START_DATE keeps every model's loss series aligned on the same
# forecast dates — a requirement for the MCS test.
START_DATE = d_gvz.index[WINDOW]


def _recency_weights(n, delta):
    """Geometric recency weights for a rolling window of length n.

    Newest row (last in the window) gets weight 1; the row `age` steps older gets
    delta**age. delta=1.0 -> equal weights (the unweighted base case). Weights are
    normalised to mean 1 (sum = n) so they only re-balance the fit *relative* to
    each other — keeping the QLIKE comparable across delta.
    """
    if delta >= 1.0:
        return np.ones(n)
    ages = np.arange(n)[::-1]          # oldest = n-1 ... newest = 0
    w = delta ** ages
    return w * (n / w.sum())           # mean-1 normalise

def _qlike(actual, forecast, eps=EPS):
    f = np.maximum(forecast, eps)
    r = actual / f
    return r - np.log(r) - 1.0, int((forecast <= eps).sum())


# --- Per-day QLIKE loss series (date-indexed) for the MCS test ------------------
# Vectorised weighted-OLS log+smearing eval, common-OOS gated. Plain OLS = WLS with
# positional weights from _recency_weights(window, delta). Returns the per-forecast
# -day QLIKE loss as a date-indexed Series (forecast-origin dates), one per model.
def rolling_log_ols_loss_series(design, feat_cols, window, start_date=None, delta=1.0):
    if start_date is None:
        start_date = START_DATE
    X = np.column_stack([np.ones(len(design)), design[feat_cols].to_numpy()])
    yl  = design["y_log"].to_numpy()
    lvl = design["y_level"].to_numpy()
    idx = design.index
    N, p = X.shape
    t_all = np.arange(window, N)
    t_oos = t_all[idx[window:] >= start_date]
    starts = t_oos - window
    Xwins = np.lib.stride_tricks.sliding_window_view(X, window, axis=0)[starts].transpose(0, 2, 1)
    ywins = np.lib.stride_tricks.sliding_window_view(yl, window)[starts]
    w  = _recency_weights(window, delta); sw = np.sqrt(w)
    Xs = Xwins * sw[None, :, None]; ys = ywins * sw[None, :]
    A = np.einsum("nwi,nwj->nij", Xs, Xs)
    b = np.einsum("nwi,nw->ni", Xs, ys)
    beta = np.linalg.solve(A, b)
    fitted = np.einsum("nwp,np->nw", Xwins, beta)
    smear = np.einsum("nw,w->n", np.exp(ywins - fitted), w) / w.sum()
    x_pred = np.einsum("np,np->n", X[t_oos], beta)
    fc = np.exp(x_pred) * smear
    ac = lvl[t_oos]
    q, _ = _qlike(ac, fc)
    return pd.Series(q, index=idx[t_oos], name="qlike")


print(f"Common OOS start: {START_DATE.date()}  "
      f"({int((d_gvz.index >= START_DATE).sum())} forecast days)")

Common OOS start: 2016-07-11  (2481 forecast days)


In [4]:
# ===========================================================================
# Cell 4 — Run the 6-model MCS and save to iv_comparison.csv
# ===========================================================================
# Build one per-day QLIKE loss series per model at the fixed (WINDOW, DELTA), align
# on the common OOS dates, and run the Model Confidence Set (Hansen, Lunde & Nason
# 2011) via arch.bootstrap.MCS. p > 0.05 => kept in the 5% confidence set; p <= 0.05
# => significantly inferior to the best model. The three IV-only specs (gold GVZ vs
# equity VIX vs crude OVX) answer whether cross-asset IV forecasts gold vol as well as
# gold-specific IV, or whether GVZ is genuinely better.
from arch.bootstrap import MCS

models = [
    ("log HAR + GVZ",              d_gvz,         ["x_d", "x_w", "x_m", "log_GVZ"]),
    ("log HAR + SPX IV (VIX)",     d_vix,         ["x_d", "x_w", "x_m", "log_VIX"]),
    ("log HAR + Crude IV (OVX)",   d_ovx,         ["x_d", "x_w", "x_m", "log_OVX"]),
    ("log HAR + Macro",            d_macro,       ["x_d", "x_w", "x_m", "macro"]),
    ("log HAR + Macro + SPX RV",   d_macro_spx,   ["x_d", "x_w", "x_m", "macro", "log_RV_ES"]),
    ("log HAR + Macro + Crude RV", d_macro_crude, ["x_d", "x_w", "x_m", "macro", "log_RV_crude"]),
]

loss_cols = {name: rolling_log_ols_loss_series(d, f, WINDOW, START_DATE, DELTA)
             for name, d, f in models}
losses = pd.DataFrame(loss_cols).dropna()
assert losses.notna().all().all() and len(losses) > 0
print(f"MCS loss matrix: {losses.shape[0]} OOS days x {losses.shape[1]} models "
      f"({losses.index.min().date()} .. {losses.index.max().date()})\n")

mcs = MCS(losses, size=0.05, reps=10000, block_size=None,
          method="max", bootstrap="stationary", seed=42)
mcs.compute()

pv = mcs.pvalues["Pvalue"]
mcs_results = (pd.DataFrame({
        "model": list(losses.columns),
        "mean_qlike": [losses[c].mean() for c in losses.columns],
        "pvalue": [float(pv[c]) for c in losses.columns],
        "in_mcs": [bool(pv[c] > 0.05) for c in losses.columns],
    })
    .sort_values("pvalue", ascending=False).reset_index(drop=True))
mcs_results.to_csv("iv_comparison.csv", index=False)

pd.set_option("display.float_format", lambda v: f"{v:.6f}")
print(f"{int(mcs_results['in_mcs'].sum())} of {len(mcs_results)} models in the 5% MCS; "
      f"{int((~mcs_results['in_mcs']).sum())} excluded (p<=0.05).")
print(f"Lowest mean-QLIKE model: {losses.mean().idxmin()}  ({losses.mean().min():.6f})")
print("Saved -> iv_comparison.csv\n")
mcs_results

MCS loss matrix: 2481 OOS days x 6 models (2016-07-11 .. 2026-05-28)



1 of 6 models in the 5% MCS; 5 excluded (p<=0.05).
Lowest mean-QLIKE model: log HAR + GVZ  (0.028842)
Saved -> iv_comparison.csv



,model,mean_qlike,pvalue,in_mcs
0,log HAR + GVZ,0.028842,1.000000,True
1,log HAR + SPX IV (VIX),0.031794,0.008500,False
2,log HAR + Macro,0.031373,0.008500,False
3,log HAR + Macro + SPX RV,0.031215,0.008500,False
4,log HAR + Macro + Crude RV,0.031335,0.008500,False
5,log HAR + Crude IV (OVX),0.032065,0.000000,False
